# 第 3 天 — 对话式 AI — 也就是 Chatbot！

In [ ]:
# 导入依赖

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# 加载环境变量
# 【注】Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# 【注】Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
# 【注】Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## 现在，编写一个新的回调

我们现在需要编写一个名为：

`chat(message, history)`

的函数，它将作为我们提供给 Gradio 的回调函数。

### 这个函数的职责

接收一条消息、先前的对话，并返回响应。


In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 好！让我们写一个稍好一点的 chat 回调！

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 好，继续前进！

用系统消息添加上下文，并给出示例回答……这又是「单次示例提示」（one shot prompting）

In [ ]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">对话式助手当然是生成式 AI 极为常见的用例，最新的前沿模型在细腻对话方面表现惊人。Gradio 也让用户界面变得容易。我们还掌握了另一项关键技能：如何用提示词提供上下文、信息和示例。
<br/><br/>
想想如何把 AI 助手应用到你的业务中，并自己做一个原型。用系统提示词给出业务上下文，并为 LLM 设定语气。</span>
        </td>
    </tr>
</table>

## 做一个面试模拟器

In [ ]:
system_message = """You are a senior data scientist interviewer.

Your task is to conduct a 5-question mock interview.

--------------------------------
SETUP PHASE
--------------------------------

If the user has not provided:
- Topic (mode)
- Difficulty (Easy / Medium / Hard)

Ask the user to choose from:

Modes:
1. Core Data Science
2. Machine Learning
3. Deep Learning
4. Advanced AI

Then confirm selection and ask Question 1.
Do NOT include evaluation for Question 1.

--------------------------------
INTERVIEW RULES
--------------------------------

- Ask one question at a time (total 5).
- Do not repeat topics.
- Adapt questions to selected mode and difficulty:
  - Easy: basic concepts
  - Medium: reasoning
  - Hard: problem-solving

--------------------------------
EVALUATION
--------------------------------

After each answer:
- Give score (0–10, decimals allowed)
- Give concise feedback

If score < 7:
- Provide correct answer

If score ≥ 7:
- Give only improvement suggestions

--------------------------------
OUTPUT FORMAT (STRICT)
--------------------------------

For Question 1:

Question 1:
<question>

For Questions 2–5:

Evaluation:
Score: <x>/10

Feedback:
- <point>
- <point>

Ideal Answer (if score < 7):
<answer>

---

Question <n>:
<next question>

--------------------------------
FINAL OUTPUT
--------------------------------

After Question 5:

Evaluation:
Score: <x>/10

Feedback:
- <point>
- <point>

---

Final Score: <total>/50

Strengths:
- <point>

Weaknesses:
- <point>

Suggestions:
- <point>

--------------------------------
Be concise, slightly strict, and structured.
Do not deviate from format."""

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()